# NCAA Basketball Tournament — Modelling Pipeline

## March Machine Learning Mania 2026

This notebook trains a prediction model for NCAA tournament outcomes using the features built in Notebook 1.

**Pipeline overview:**
1. Build matchup rows (feature diffs between pairs of teams)
2. Establish a seed-difference baseline
3. Train LightGBM with Optuna hyperparameter search
4. Collect out-of-fold predictions
5. Stack with a LogReg meta-learner
6. Generate a submission CSV

**Key design choice — LOSO-CV:** We use Leave-One-Season-Out cross-validation: each fold withholds one full tournament year (~67 games) and trains on all prior years. This matches the test distribution (predict a single future tournament) much better than random k-fold.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA = Path("/kaggle/input/march-machine-learning-mania-2026")

# ── Helper: Brier Score ────────────────────────────────────────────────────
def brier_score(y_true, y_pred):
    """MSE of probability predictions. Lower is better."""
    return float(np.mean((np.clip(y_pred, 0, 1) - y_true) ** 2))

# ── Load Men's data ────────────────────────────────────────────────────────
compact_M  = pd.read_csv(DATA / "MRegularSeasonCompactResults.csv")
detailed_M = pd.read_csv(DATA / "MRegularSeasonDetailedResults.csv")
massey     = pd.read_csv(DATA / "MMasseyOrdinals.csv")
tourney_M  = pd.read_csv(DATA / "MNCAATourneyCompactResults.csv")
seeds_M    = pd.read_csv(DATA / "MNCAATourneySeeds.csv")
seeds_M["SeedNum"] = seeds_M["Seed"].str[1:3].astype(int)

# ── Load Women's data ──────────────────────────────────────────────────────
compact_W  = pd.read_csv(DATA / "WRegularSeasonCompactResults.csv")
detailed_W = pd.read_csv(DATA / "WRegularSeasonDetailedResults.csv")
tourney_W  = pd.read_csv(DATA / "WNCAATourneyCompactResults.csv")
seeds_W    = pd.read_csv(DATA / "WNCAATourneySeeds.csv")
seeds_W["SeedNum"] = seeds_W["Seed"].str[1:3].astype(int)

print("Men's data:")
print(f"  Regular season games: {len(compact_M):,}")
print(f"  Tournament games:     {len(tourney_M):,}")
print(f"  Massey rankings:      {len(massey):,}")
print("\nWomen's data:")
print(f"  Regular season games: {len(compact_W):,}")
print(f"  Tournament games:     {len(tourney_W):,}")

In [ ]:
# ── Feature-building functions (same as Notebook 1) ───────────────────────

def build_efficiency_features(compact_df, detailed_df):
    """Aggregate box-score stats into per-(Season, TeamID) efficiency metrics."""
    det = detailed_df.copy()
    win_rows = pd.DataFrame({
        "Season": det.Season, "TeamID": det.WTeamID,
        "own_pts": det.WScore, "opp_pts": det.LScore,
        "FGM": det.WFGM, "FGA": det.WFGA, "FGM3": det.WFGM3,
        "FTA": det.WFTA, "OR": det.WOR, "DR": det.WDR, "TO": det.WTO,
        "oFGA": det.LFGA, "oOR": det.LOR, "oTO": det.LTO, "oFTA": det.LFTA, "oDR": det.LDR,
        "won": 1,
    })
    loss_rows = pd.DataFrame({
        "Season": det.Season, "TeamID": det.LTeamID,
        "own_pts": det.LScore, "opp_pts": det.WScore,
        "FGM": det.LFGM, "FGA": det.LFGA, "FGM3": det.LFGM3,
        "FTA": det.LFTA, "OR": det.LOR, "DR": det.LDR, "TO": det.LTO,
        "oFGA": det.WFGA, "oOR": det.WOR, "oTO": det.WTO, "oFTA": det.WFTA, "oDR": det.WDR,
        "won": 0,
    })
    games = pd.concat([win_rows, loss_rows], ignore_index=True)
    games["margin"] = games["own_pts"] - games["opp_pts"]
    grp  = games.groupby(["Season", "TeamID"])
    sums = grp[["own_pts","opp_pts","FGA","FGM","FGM3","FTA","OR","DR","TO",
                "oFGA","oOR","oTO","oFTA","oDR"]].sum().reset_index()
    base = grp.agg(win_pct=("won","mean"), avg_margin=("margin","mean")).reset_index()
    agg  = base.merge(sums, on=["Season","TeamID"])
    own_poss = (agg.FGA - agg.OR  + agg.TO  + 0.44*agg.FTA).replace(0, np.nan)
    opp_poss = (agg.oFGA - agg.oOR + agg.oTO + 0.44*agg.oFTA).replace(0, np.nan)
    agg["off_eff"]  = agg.own_pts / own_poss * 100
    agg["def_eff"]  = agg.opp_pts / opp_poss * 100
    agg["net_eff"]  = agg.off_eff - agg.def_eff
    agg["efg_pct"]  = (agg.FGM + 0.5*agg.FGM3) / agg.FGA
    agg["oreb_pct"] = agg.OR  / (agg.OR  + agg.oDR)
    agg["dreb_pct"] = agg.DR  / (agg.DR  + agg.oOR)
    agg["to_pct"]   = agg.TO  / own_poss
    agg["ft_rate"]  = agg.FTA / agg.FGA
    return agg[["Season","TeamID","win_pct","avg_margin","net_eff","off_eff","def_eff",
                "efg_pct","oreb_pct","dreb_pct","to_pct","ft_rate"]].copy()

def build_elo_features(compact_df):
    """Margin-of-victory Elo with season carryover (75/25 mean reversion)."""
    seasons = sorted(compact_df["Season"].unique())
    elo = {}
    records = []
    for season in seasons:
        season_games = compact_df[compact_df["Season"] == season].sort_values("DayNum")
        teams = set(season_games["WTeamID"]) | set(season_games["LTeamID"])
        for tid in teams:
            elo[tid] = elo[tid] * 0.75 + 1500 * 0.25 if tid in elo else 1500.0
        k_wins = {tid: 0.0 for tid in teams}
        for row in season_games.itertuples(index=False):
            w, l   = int(row.WTeamID), int(row.LTeamID)
            margin = max(float(row.WScore - row.LScore), 1.0)
            ew, el = elo.get(w, 1500.0), elo.get(l, 1500.0)
            exp_w  = 1 / (1 + 10 ** ((el - ew) / 400))
            k_eff  = 20 * (1 + margin / 20) ** 0.6
            delta  = k_eff * (1 - exp_w)
            elo[w] += delta; elo[l] -= delta
            k_wins[w] += k_eff
        for tid in teams:
            records.append({"Season": season, "TeamID": tid,
                            "elo_rating": elo[tid], "elo_k_weighted_wins": k_wins[tid]})
    return pd.DataFrame(records)

def build_massey_pca_features(massey_df):
    """PCA across all Massey systems with >= 50% coverage. Men's only."""
    df = massey_df[massey_df["RankingDayNum"] <= 133].copy()
    df = df.sort_values("RankingDayNum")
    last = df.groupby(["Season","TeamID","SystemName"])["OrdinalRank"].last().reset_index()
    records = []
    for season in sorted(last["Season"].unique()):
        sdf   = last[last["Season"] == season]
        pivot = sdf.pivot_table(index="TeamID", columns="SystemName",
                                values="OrdinalRank", aggfunc="first")
        coverage = pivot.notna().sum() / len(pivot)
        keep = coverage[coverage >= 0.5].index.tolist()
        if len(keep) < 2:
            for tid in pivot.index:
                records.append({"Season": season, "TeamID": int(tid),
                                "massey_pc1": np.nan, "massey_pc2": np.nan})
            continue
        pivot = pivot[keep]
        pivot = pivot.max().max() - pivot + 1
        pivot_filled = pivot.fillna(pivot.mean())
        mat = StandardScaler().fit_transform(pivot_filled)
        n_comp = min(2, mat.shape[1])
        pca = PCA(n_components=n_comp, random_state=42)
        pcs = pca.fit_transform(mat)
        row_means = pivot.mean(axis=1).values
        top_q_idx = np.where(row_means > np.percentile(row_means, 75))[0]
        if len(top_q_idx) > 0 and pcs[top_q_idx, 0].mean() < 0:
            pcs[:, 0] = -pcs[:, 0]
        for i, tid in enumerate(pivot.index):
            records.append({"Season": season, "TeamID": int(tid),
                            "massey_pc1": float(pcs[i, 0]),
                            "massey_pc2": float(pcs[i, 1]) if n_comp >= 2 else np.nan})
    return pd.DataFrame(records)

def build_momentum_features(compact_df, n_games=10):
    """Last-N-games momentum. Excludes conf tourney (DayNum >= 132)."""
    comp = compact_df[compact_df["DayNum"] < 132].copy()
    win_rows  = comp[["Season","DayNum","WTeamID","WScore","LScore"]].copy()
    win_rows["TeamID"] = win_rows["WTeamID"]; win_rows["won"] = 1
    win_rows["margin"] = win_rows["WScore"] - win_rows["LScore"]
    loss_rows = comp[["Season","DayNum","LTeamID","WScore","LScore"]].copy()
    loss_rows["TeamID"] = loss_rows["LTeamID"]; loss_rows["won"] = 0
    loss_rows["margin"] = loss_rows["LScore"] - loss_rows["WScore"]
    games = pd.concat([win_rows[["Season","DayNum","TeamID","won","margin"]],
                       loss_rows[["Season","DayNum","TeamID","won","margin"]]],
                      ignore_index=True).sort_values(["Season","TeamID","DayNum"])
    last_n = games.groupby(["Season","TeamID"]).tail(n_games)
    agg = last_n.groupby(["Season","TeamID"]).agg(
        recent_win_pct=("won","mean"), recent_net_margin=("margin","mean")).reset_index()
    def compute_streak(grp):
        outcomes = grp.sort_values("DayNum")["won"].values
        if len(outcomes) == 0: return 0
        val, s = outcomes[-1], 0
        for o in reversed(outcomes):
            if o == val: s += 1
            else: break
        return s if val == 1 else -s
    streak = (games.groupby(["Season","TeamID"])
              .apply(compute_streak, include_groups=False)
              .reset_index(name="streak"))
    result = agg.merge(streak, on=["Season","TeamID"], how="left")
    result["streak"] = result["streak"].fillna(0).astype(int)
    return result

# ── Build Men's team features ──────────────────────────────────────────────
print("Building Men's features (~2-3 min)...")
team_features_M = (
    build_efficiency_features(compact_M, detailed_M)
    .merge(build_elo_features(compact_M),          on=["Season","TeamID"], how="left")
    .merge(build_massey_pca_features(massey),       on=["Season","TeamID"], how="left")
    .merge(build_momentum_features(compact_M),      on=["Season","TeamID"], how="left")
    .merge(seeds_M[["Season","TeamID","SeedNum"]], on=["Season","TeamID"], how="left")
)

# ── Build Women's team features ────────────────────────────────────────────
print("Building Women's features (~2 min)...")
team_features_W = (
    build_efficiency_features(compact_W, detailed_W)
    .merge(build_elo_features(compact_W),          on=["Season","TeamID"], how="left")
    .merge(build_momentum_features(compact_W),     on=["Season","TeamID"], how="left")
    .merge(seeds_W[["Season","TeamID","SeedNum"]], on=["Season","TeamID"], how="left")
)
# Women's have no Massey rankings — fill with NaN (imputer handles this downstream)
team_features_W["massey_pc1"] = np.nan
team_features_W["massey_pc2"] = np.nan

print(f"\nMen's team features:   {team_features_M.shape}")
print(f"Women's team features: {team_features_W.shape}")

## 2. Building Matchup Rows

Each training example is one tournament game. We represent it as:

```
feature_diff = team1_feature − team2_feature
```

Where Team1 = lower TeamID, Team2 = higher TeamID. Label = 1 if Team1 won.

This antisymmetric encoding means the model sees the same information regardless of which team is "first" — flipping the sign of all diffs gives the prediction for the reverse matchup.

We use a **NaN-tolerant** matchup builder: rows are only dropped if both teams are absent from the feature set. Partial NaN (e.g. Massey unavailable for older seasons) is kept and filled by a SimpleImputer during training.

In [ ]:
CURATED = {
    "SeedNum", "net_eff", "efg_pct", "oreb_pct", "dreb_pct", "to_pct", "ft_rate",
    "win_pct", "avg_margin", "elo_rating", "elo_k_weighted_wins",
    "massey_pc1", "massey_pc2",
    "recent_win_pct", "recent_net_margin", "streak",
}

def build_matchup_df(tourney_df, features_df):
    feat_cols = [c for c in features_df.columns if c not in ("Season","TeamID")]
    df = tourney_df.copy()
    df["Team1ID"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["Team2ID"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"]   = (df["WTeamID"] == df["Team1ID"]).astype(int)
    feat = features_df.set_index(["Season","TeamID"])
    t1 = (df[["Season","Team1ID"]].rename(columns={"Team1ID":"TeamID"})
          .join(feat, on=["Season","TeamID"]).drop(columns="TeamID"))
    t2 = (df[["Season","Team2ID"]].rename(columns={"Team2ID":"TeamID"})
          .join(feat, on=["Season","TeamID"]).drop(columns="TeamID"))
    result = df[["Season","Team1ID","Team2ID","Label"]].copy()
    for c in feat_cols:
        result[f"{c}_diff"] = t1[c].values - t2[c].values
    return result.reset_index(drop=True)

all_matchups_M = build_matchup_df(tourney_M, team_features_M)
all_matchups_W = build_matchup_df(tourney_W, team_features_W)

# Feature columns per gender — Women's exclude Massey (all NaN, no signal)
FEAT_COLS_M = [c for c in all_matchups_M.columns
               if c.endswith("_diff") and c.replace("_diff","") in CURATED]
FEAT_COLS_W = [c for c in all_matchups_W.columns
               if c.endswith("_diff") and c.replace("_diff","") in CURATED
               and "massey" not in c]

# CV seasons: last 10 tournament seasons excluding 2020 (cancelled)
cv_seasons_M = [s for s in sorted(tourney_M["Season"].unique()) if s != 2020][-10:]
cv_seasons_W = [s for s in sorted(tourney_W["Season"].unique()) if s != 2020][-10:]

print(f"Men's   matchup rows: {len(all_matchups_M):,}  |  Features: {len(FEAT_COLS_M)}")
print(f"Women's matchup rows: {len(all_matchups_W):,}  |  Features: {len(FEAT_COLS_W)}")
print(f"Men's CV seasons:   {cv_seasons_M}")
print(f"Women's CV seasons: {cv_seasons_W}")

## 3. Seed-Difference Baseline

Before any ML, let's establish what "free information" gives us.

A simple logistic regression trained only on `SeedNum_diff` achieves Brier ≈ 0.196. This is our floor — any model that can't beat this isn't learning anything useful beyond seeds.

Lower seeds (1, 2) beat higher seeds (15, 16) about 95% of the time historically. But seeds are noisy for mid-seed matchups (8 vs 9 is essentially a coin flip). Net efficiency and Elo help discriminate these.

In [ ]:
def run_loso_cv(model_fn, feat_cols, all_matchups, cv_seasons):
    """Leave-One-Season-Out CV. model_fn(X_train, y_train) → fitted model."""
    oof_preds, oof_labels = [], []
    for val_season in cv_seasons:
        train = all_matchups[all_matchups["Season"] != val_season]
        val   = all_matchups[all_matchups["Season"] == val_season]
        X_tr, y_tr = train[feat_cols].values, train["Label"].values
        X_va, y_va = val[feat_cols].values,   val["Label"].values

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_va = imp.transform(X_va)

        model = model_fn(X_tr, y_tr)
        preds = model.predict_proba(X_va)[:, 1]
        oof_preds.extend(preds)
        oof_labels.extend(y_va)

    return brier_score(np.array(oof_labels), np.array(oof_preds))

# Men's seed baseline (illustrative — Women's analogous)
seed_cols_M = [c for c in FEAT_COLS_M if "SeedNum" in c]
brier_seed_M = run_loso_cv(
    lambda X, y: LogisticRegression(C=1e9).fit(X, y),
    seed_cols_M, all_matchups_M, cv_seasons_M
)
print(f"Men's seed baseline LOSO-CV Brier: {brier_seed_M:.4f}")

## 4. LightGBM with Optuna Hyperparameter Search

LightGBM (gradient boosted trees) consistently outperforms logistic regression on tabular data with non-linear feature interactions.

**Key hyperparameters we tune:**
- `num_leaves` — tree complexity (default 31; we search 10–150)
- `learning_rate` — step size (we search 0.01–0.3)
- `min_child_samples` — regularisation via minimum leaf size (search 10–100)
- `reg_lambda`, `reg_alpha` — L2/L1 regularisation

**Optuna uses Tree-structured Parzen Estimation (TPE):** Unlike grid search or random search, TPE builds a probabilistic model of which hyperparameter regions are promising and samples from them. It typically finds good solutions in 30–50 trials vs hundreds for grid search.

In [ ]:
def lgbm_objective(trial, feat_cols, all_matchups, cv_seasons):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000),
        "num_leaves":        trial.suggest_int("num_leaves", 10, 150),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective": "binary", "verbose": -1, "n_jobs": -1,
    }
    oof_preds, oof_labels = [], []
    for val_season in cv_seasons:
        train = all_matchups[all_matchups["Season"] != val_season]
        val   = all_matchups[all_matchups["Season"] == val_season]
        X_tr, y_tr = train[feat_cols].values, train["Label"].values
        X_va, y_va = val[feat_cols].values,   val["Label"].values
        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr); X_va = imp.transform(X_va)
        model = lgb.LGBMClassifier(**params)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)
        oof_preds.extend(model.predict_proba(X_va)[:, 1])
        oof_labels.extend(y_va)
    return brier_score(np.array(oof_labels), np.array(oof_preds))

# ── Tune Men's model ───────────────────────────────────────────────────────
print("Tuning Men's model (30 trials)...")
study_M = optuna.create_study(direction="minimize")
study_M.optimize(
    lambda t: lgbm_objective(t, FEAT_COLS_M, all_matchups_M, cv_seasons_M),
    n_trials=30, show_progress_bar=True
)
best_params_M = study_M.best_params
print(f"\nBest Men's LGBM Brier: {study_M.best_value:.4f}")

# ── Tune Women's model ─────────────────────────────────────────────────────
print("\nTuning Women's model (20 trials)...")
study_W = optuna.create_study(direction="minimize")
study_W.optimize(
    lambda t: lgbm_objective(t, FEAT_COLS_W, all_matchups_W, cv_seasons_W),
    n_trials=20, show_progress_bar=True
)
best_params_W = study_W.best_params
print(f"\nBest Women's LGBM Brier: {study_W.best_value:.4f}")

## 5. Ensemble + Meta-Learner

A single model's predictions can be overconfident in certain matchup types. Averaging predictions from multiple models with different inductive biases reduces this variance.

**Meta-learner (stacking):** Instead of a fixed-weight average, we train a logistic regression on **out-of-fold predictions** — predictions generated by the base model on data it never saw during training. This prevents the meta-learner from memorising the most confident base model.

We transform base predictions to **logit space** (log-odds) before feeding them to the meta-learner. This is natural because logistic regression is linear in log-odds — the native scale for combining calibrated probability estimates.

**Why C=1.0?** With correlated inputs (boosted models tend to agree ~80% of the time), strong L2 regularisation prevents weights from diverging to compensate for multicollinearity.

In [ ]:
def collect_oof(best_params, feat_cols, all_matchups, cv_seasons):
    """Collect out-of-fold predictions using best Optuna hyperparameters."""
    oof_df = all_matchups[all_matchups["Season"].isin(cv_seasons)][
        ["Season","Label"]].copy().reset_index(drop=True)
    oof_df["pred"] = np.nan

    for val_season in cv_seasons:
        train = all_matchups[all_matchups["Season"] != val_season]
        val   = all_matchups[all_matchups["Season"] == val_season]
        X_tr, y_tr = train[feat_cols].values, train["Label"].values
        X_va       = val[feat_cols].values

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr); X_va = imp.transform(X_va)

        model = lgb.LGBMClassifier(**best_params, objective="binary", verbose=-1)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)
        mask = oof_df["Season"] == val_season
        oof_df.loc[mask, "pred"] = model.predict_proba(X_va)[:, 1]

    return oof_df

EPS = 1e-7
def to_logit(p):
    p = np.clip(p, EPS, 1-EPS)
    return np.log(p / (1 - p))

def fit_meta(oof_df):
    """Fit LogReg meta-learner on OOF predictions in logit space."""
    y_true = oof_df["Label"].values
    L      = to_logit(oof_df["pred"].values).reshape(-1, 1)
    meta   = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000)
    meta.fit(L, y_true)
    brier  = brier_score(y_true, meta.predict_proba(L)[:, 1])
    return meta, brier

# ── Men's OOF + meta ───────────────────────────────────────────────────────
oof_M         = collect_oof(best_params_M, FEAT_COLS_M, all_matchups_M, cv_seasons_M)
meta_M, bm_M  = fit_meta(oof_M)
print(f"Men's   OOF LGBM Brier: {brier_score(oof_M['Label'].values, oof_M['pred'].values):.4f}")
print(f"Men's   Meta Brier:     {bm_M:.4f}")

# ── Women's OOF + meta ─────────────────────────────────────────────────────
oof_W         = collect_oof(best_params_W, FEAT_COLS_W, all_matchups_W, cv_seasons_W)
meta_W, bm_W  = fit_meta(oof_W)
print(f"\nWomen's OOF LGBM Brier: {brier_score(oof_W['Label'].values, oof_W['pred'].values):.4f}")
print(f"Women's Meta Brier:     {bm_W:.4f}")

## 6. Generating the Submission

**Stage 1** (`SampleSubmissionStage1.csv`) covers all possible matchups for seasons **2022–2025** (~519k rows). Submitting this file gives you a public leaderboard score.

**Stage 2** (`SampleSubmissionStage2.csv`) covers all possible 2026 matchups (~132k rows). This is the actual competition submission. To switch, change the filename in the cell below.

For each row we:
1. Parse the ID → Season, Team1ID, Team2ID, gender (Men's: TeamID < 3000)
2. Look up per-season team features for both teams
3. Compute feature diffs (Team1 − Team2)
4. Train the final model on **all** available tournament data
5. Apply the meta-learner (logit-space logistic regression)
6. Clip to [0.025, 0.975] — avoids extreme overconfidence at log-loss boundaries

In [ ]:
# ── Change to SampleSubmissionStage2.csv for the final 2026 predictions ───
SAMPLE_FILE = "SampleSubmissionStage1.csv"   # Stage 1: LB scoring (2022–2025)
# SAMPLE_FILE = "SampleSubmissionStage2.csv" # Stage 2: actual competition (2026)

sub = pd.read_csv(DATA / SAMPLE_FILE)
parsed      = sub["ID"].str.split("_", expand=True).astype(int)
sub["Season"]  = parsed[0]
sub["Team1ID"] = parsed[1]
sub["Team2ID"] = parsed[2]
sub["gender"]  = np.where(sub["Team1ID"] < 3000, "M", "W")
sub["Pred"]    = 0.5   # safe default — overwritten for both genders below

def predict_gender(sub_df, team_features, best_params, meta, all_matchups, feat_cols):
    """
    Train a final model on all available tournament data, then predict for sub_df.
    sub_df must have Season, Team1ID, Team2ID columns and a reset index.
    """
    feat = team_features.set_index(["Season","TeamID"])

    t1 = (sub_df[["Season","Team1ID"]].rename(columns={"Team1ID":"TeamID"})
          .join(feat, on=["Season","TeamID"]).drop(columns="TeamID"))
    t2 = (sub_df[["Season","Team2ID"]].rename(columns={"Team2ID":"TeamID"})
          .join(feat, on=["Season","TeamID"]).drop(columns="TeamID"))

    raw_cols = [c.replace("_diff", "") for c in feat_cols]
    X_sub = np.column_stack([t1[rc].values - t2[rc].values for rc in raw_cols])

    # Retrain on ALL tournament data (CV already told us the expected performance)
    X_all = all_matchups[feat_cols].values
    y_all = all_matchups["Label"].values
    imp   = SimpleImputer(strategy="median")
    X_all_imp = imp.fit_transform(X_all)
    X_sub_imp = imp.transform(X_sub)

    params = {**best_params, "objective": "binary", "verbose": -1}
    final_model = lgb.LGBMClassifier(**params)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        final_model.fit(X_all_imp, y_all)

    base_pred = final_model.predict_proba(X_sub_imp)[:, 1]
    meta_pred = meta.predict_proba(to_logit(base_pred).reshape(-1, 1))[:, 1]
    return np.clip(meta_pred, 0.025, 0.975)

# ── Men's predictions ──────────────────────────────────────────────────────
sub_m  = sub[sub["gender"] == "M"].copy().reset_index(drop=True)
pred_M = predict_gender(sub_m, team_features_M, best_params_M, meta_M,
                        all_matchups_M, FEAT_COLS_M)
sub.loc[sub["gender"] == "M", "Pred"] = pred_M

# ── Women's predictions ────────────────────────────────────────────────────
sub_w  = sub[sub["gender"] == "W"].copy().reset_index(drop=True)
pred_W = predict_gender(sub_w, team_features_W, best_params_W, meta_W,
                        all_matchups_W, FEAT_COLS_W)
sub.loc[sub["gender"] == "W", "Pred"] = pred_W

# ── Save ───────────────────────────────────────────────────────────────────
sub[["ID","Pred"]].to_csv("/kaggle/working/submission.csv", index=False)
print(f"Submission saved: {len(sub):,} rows")
print(f"Men's   Pred range: [{pred_M.min():.3f}, {pred_M.max():.3f}]  mean={pred_M.mean():.3f}")
print(f"Women's Pred range: [{pred_W.min():.3f}, {pred_W.max():.3f}]  mean={pred_W.mean():.3f}")
print(sub[["ID","Pred"]].head())